# Cleaning Dataset Cafe Kalimantan Selatan

Notebook ini membersihkan data hasil scraping Google Maps (`cafe_scrap.csv`) dengan langkah:

1. Pilih hanya kolom: `place_id, title, category, address, latitude, longitude, open_hours, popular_times, review_count, review_rating`
2. Buang baris dengan alamat di luar Kalimantan Selatan
3. Hapus data duplikat
4. Hanya simpan kategori: `Cafe`, `Coffee shop`, `Restaurant`, `Convenience store`
5. Ekstrak `popular_times` menjadi `peak_day` dan `peak_hour`
6. Ekspor ke `clean_dataset_cafe_kalsel.csv`

Letakkan `cafe_scrap.csv` di folder yang sama dengan notebook ini (atau ubah `INPUT_FILE` di bawah).

In [1]:
import json
import re

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

# ====== KONFIGURASI ======
INPUT_FILE = "cafe_scrap.csv"
OUTPUT_FILE = "clean_dataset_cafe_kalsel.csv"

KEEP_COLUMNS = [
    "place_id", "title", "category", "address",
    "latitude", "longitude",
    "open_hours", "popular_times",
    "review_count", "review_rating",
]

# Kategori yang dipertahankan (dicocokkan persis, tidak peka huruf besar/kecil)
KEEP_CATEGORIES = ["cafe", "coffee shop", "restaurant", "convenience store"]

# Pola penanda provinsi Kalimantan Selatan di alamat (Indonesia / Inggris)
KALSEL_PATTERN = r"kalimantan selatan|south kalimantan|kalsel"

# Urutan hari (untuk menentukan peak_day bila ada nilai kembar)
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

## 0. Muat data mentah

In [2]:
raw = pd.read_csv(INPUT_FILE, dtype=str)   # semua dibaca sebagai teks dulu, tipe data diatur di akhir
print("Data mentah:", raw.shape[0], "baris x", raw.shape[1], "kolom")
raw.head(3)

Data mentah: 1252 baris x 36 kolom


,input_id,link,title,category,address,open_hours,popular_times,website,phone,plus_code,...,reservations,order_online,menu,owner,complete_address,credit_cards_accepted,about,user_reviews,user_reviews_extended,emails
0,3043499d-f7a0-4139-b121-07920cff8870,https://www.google.com/maps/place/Kise+Coffee/data=!4m7!3m6!1s0x2de423b110e2...,Kise Coffee,Coffee shop,"Melayu (Di Arena Suzuki Bagian Kiri, Jl. A. Yani km 2 No.6, Melayu, Kec. Ban...","{""Friday"":[""9.00 am–12.00 am""],""Monday"":[""9.00 am–11.00 pm""],""Saturday"":[""9....","{""Friday"":{""10"":68,""11"":81,""12"":84,""13"":75,""14"":78,""15"":71,""16"":75,""17"":68,""...",NaN,NaN,"MJG3+69 Melayu, Banjarmasin City, South Kalimantan",...,NaN,NaN,"{""link"":"""",""source"":""""}","{""id"":""113214016193856501526"",""name"":""Kise Coffee (Owner)"",""link"":""https://w...","{""borough"":""Melayu, Banjarmasin Tengah"",""street"":""Melayu (Di Arena Suzuki Ba...",NaN,"[{""id"":""service_options"",""name"":""Service options"",""options"":[{""name"":""Outdoo...","[{""Name"":""fa tamb"",""ProfilePicture"":""https://lh3.googleusercontent.com/a/ACg...",NaN,NaN
1,3043499d-f7a0-4139-b121-07920cff8870,https://www.google.com/maps/place/Berdua+Coffee/data=!4m7!3m6!1s0x2de423005a...,Berdua Coffee,Coffee shop,"Jl. A. Yani No.147, Karang Mekar, Kec. Banjarmasin Tim., Kota Banjarmasin, K...","{""Friday"":[""7.30 pm–2.00 am""],""Monday"":[""7.30 pm–2.00 am""],""Saturday"":[""7.30...",{},NaN,NaN,"MJC5+R4 Karang Mekar, Banjarmasin City, South Kalimantan",...,NaN,NaN,"{""link"":"""",""source"":""""}","{""id"":"""",""name"":""Berdua Coffee (Owner)"",""link"":""""}","{""borough"":""Karang Mekar, Banjarmasin Timur"",""street"":""Jl. A. Yani No.147"",""...",NaN,"[{""id"":""offerings"",""name"":""Offerings"",""options"":[{""name"":""Coffee"",""enabled"":...","[{""Name"":""Zainur Rahmani"",""ProfilePicture"":""https://lh3.googleusercontent.co...",NaN,NaN
2,3043499d-f7a0-4139-b121-07920cff8870,https://www.google.com/maps/place/Caffeinated/data=!4m7!3m6!1s0x2de42300441b...,Caffeinated,Coffee shop,"Jl. Manggis No.19a, RT.17/RW.02, Kuripan, Kec. Banjarmasin Tim., Kota Banjar...","{""Friday"":[""3.00 pm–1.00 am""],""Monday"":[""3.00 pm–12.00 am""],""Saturday"":[""3.0...",{},NaN,0878-1224-8425,"MJH6+24 Kuripan, Banjarmasin City, South Kalimantan",...,NaN,NaN,"{""link"":"""",""source"":""""}","{""id"":"""",""name"":""Caffeinated (Owner)"",""link"":""""}","{""borough"":""Kuripan, Banjarmasin Timur"",""street"":""Jl. Manggis No.19a, RT.17/...",NaN,"[{""id"":""service_options"",""name"":""Service options"",""options"":[{""name"":""Outdoo...","[{""Name"":""Siti Nurana"",""ProfilePicture"":""https://lh3.googleusercontent.com/a...",NaN,NaN


## 1. Pilih kolom yang dibutuhkan

In [3]:
missing = [c for c in KEEP_COLUMNS if c not in raw.columns]
assert not missing, f"Kolom tidak ditemukan di file: {missing}"

df = raw[KEEP_COLUMNS].copy()
print("Setelah pilih kolom:", df.shape)
df.head(3)

Setelah pilih kolom: (1252, 10)


,place_id,title,category,address,latitude,longitude,open_hours,popular_times,review_count,review_rating
0,ChIJHWziELEj5C0RjTyKdw1DrwA,Kise Coffee,Coffee shop,"Melayu (Di Arena Suzuki Bagian Kiri, Jl. A. Yani km 2 No.6, Melayu, Kec. Ban...",-3.324391,114.603467,"{""Friday"":[""9.00 am–12.00 am""],""Monday"":[""9.00 am–11.00 pm""],""Saturday"":[""9....","{""Friday"":{""10"":68,""11"":81,""12"":84,""13"":75,""14"":78,""15"":71,""16"":75,""17"":68,""...",153,4.800000
1,ChIJr_tWWgAj5C0RXmWeCegoXgY,Berdua Coffee,Coffee shop,"Jl. A. Yani No.147, Karang Mekar, Kec. Banjarmasin Tim., Kota Banjarmasin, K...",-3.327958,114.607845,"{""Friday"":[""7.30 pm–2.00 am""],""Monday"":[""7.30 pm–2.00 am""],""Saturday"":[""7.30...",{},1,5.000000
2,ChIJFRIbRAAj5C0RcjjW-dnqaic,Caffeinated,Coffee shop,"Jl. Manggis No.19a, RT.17/RW.02, Kuripan, Kec. Banjarmasin Tim., Kota Banjar...",-3.322415,114.610258,"{""Friday"":[""3.00 pm–1.00 am""],""Monday"":[""3.00 pm–12.00 am""],""Saturday"":[""3.0...",{},2,5.000000


## 2. Buang baris di luar Kalimantan Selatan

Pencarian di Google Maps kadang mengembalikan tempat dengan nama mirip di provinsi lain
(misalnya "Banjar" di Jawa Barat atau "Kotabaru" di Yogyakarta). Filter dilakukan lewat teks alamat.

In [4]:
is_kalsel = df["address"].fillna("").str.contains(KALSEL_PATTERN, case=False, regex=True)

print("Baris di Kalimantan Selatan :", is_kalsel.sum())
print("Baris di luar Kalimantan Selatan (dibuang):", (~is_kalsel).sum())

# Ringkasan provinsi dari baris yang dibuang (ambil teks sebelum kode pos di akhir alamat)
dibuang = df.loc[~is_kalsel, "address"].fillna("")
print()
print(dibuang.str.extract(r",\s*([^,]+?)\s*\d{5}\s*$")[0].value_counts(dropna=False))

df = df[is_kalsel].copy()
print("\nSetelah filter Kalsel:", df.shape)

Baris di Kalimantan Selatan : 1155
Baris di luar Kalimantan Selatan (dibuang): 97

0
Special Region of Yogyakarta    34
Jawa Barat                      25
Daerah Istimewa Yogyakarta      19
West Java                       15
Bali                             4
Name: count, dtype: int64

Setelah filter Kalsel: (1155, 10)


## 3. Hapus data duplikat

- Duplikat persis dan duplikat `place_id`
- Duplikat berdasarkan nama + alamat (setelah dinormalkan: huruf kecil, spasi dirapikan)

Tempat yang hanya **berbagi koordinat** namun namanya berbeda (misalnya dua gerai di mal yang sama)
tidak dihapus otomatis karena itu biasanya tempat berbeda. Daftarnya ditampilkan untuk dicek manual.

In [5]:
n0 = len(df)

df = df.drop_duplicates()
df = df.drop_duplicates(subset="place_id", keep="first")

def norm(s):
    return re.sub(r"\s+", " ", str(s).strip().lower())

key = df["title"].map(norm) + "|" + df["address"].map(norm)
df = df[~key.duplicated(keep="first")].copy()

print(f"Duplikat dihapus: {n0 - len(df)} baris | sisa: {len(df)}")

# Pemeriksaan manual: koordinat sama, nama berbeda
koord_sama = df[df.duplicated(["latitude", "longitude"], keep=False)].sort_values(["latitude", "longitude"])
print(f"\nBaris dengan koordinat sama (cek manual, tidak dihapus): {len(koord_sama)}")
koord_sama[["title", "address", "latitude", "longitude"]]

Duplikat dihapus: 0 baris | sisa: 1155

Baris dengan koordinat sama (cek manual, tidak dihapus): 10


,title,address,latitude,longitude
805,Kebabblasan Barabai Selatan,"C98M+2HC Kebabblasan Barabai Selatan, Jl. Ganesha, Barabai Sel., Kec. Baraba...",-2.584823,115.383898
973,Ganesa caffe,"C98M+3HC Ganesa caffe, Jl. Ganesha, Barabai Sel., Kec. Barabai, Kabupaten Hu...",-2.584823,115.383898
704,UP TO YOU KDG,"Jl. Nusa Indah No.49, Kandangan Utara, Kec. Kandangan, Kabupaten Hulu Sungai...",-2.781030,115.270715
720,Cafe Teras Jahe,"Jl. Nusa Indah No.49, Kandangan Utara, Kec. Kandangan, Kabupaten Hulu Sungai...",-2.781030,115.270715
53,Kopi Kenangan - Duta Mall Banjarmasin,"Jl. A. Yani Jl. Simpang Ulin No.KM 2 Lantai GF, Melayu, Kec. Banjarmasin Ten...",-3.322822,114.602964
108,Eskopi Indonesia Cabang Duta Mall Banjarmasin,"Jl. A. Yani No.98, Melayu, Kec. Banjarmasin Tengah, Kota Banjarmasin, Kalima...",-3.322822,114.602964
279,One Taste Coffee,"GXF3+4H5 One Taste Coffee, Jl. Dharma Praja, Pd. Butun, Kec. Batulicin, Kabu...",-3.477230,115.953988
290,One Taste Coffe,"GXF3+4H5 One Taste Coffe, Jl. Dharma Praja, Pd. Butun, Kec. Batulicin, Kabup...",-3.477230,115.953988
361,Malaka Coffee & Pastry,"Jl. Pancasila, Pelaihari, Tanah Laut Regency, South Kalimantan 70812",-3.801905,114.765728
369,Matchaboy pelaihari,"Jl. Pancasila, Pelaihari, Kec. Pelaihari, Kabupaten Tanah Laut, Kalimantan S...",-3.801905,114.765728


## 4. Filter kategori

Hanya kategori `Cafe`, `Coffee shop`, `Restaurant`, dan `Convenience store` yang dipertahankan
(pencocokan persis, jadi `Seafood restaurant` atau `Bakery` ikut dibuang).

In [6]:
cat_norm = df["category"].fillna("").str.strip().str.lower()
mask = cat_norm.isin(KEEP_CATEGORIES)

print("Kategori yang dibuang:")
print(df.loc[~mask, "category"].fillna("(kosong)").value_counts().to_string())

df = df[mask].copy()
print("\nSetelah filter kategori:", df.shape)
print(df["category"].value_counts().to_string())

Kategori yang dibuang:
category
Ice cream shop                  12
Diner                            9
Seafood restaurant               7
Bar                              5
Indonesian restaurant            5
Espresso bar                     4
Bakery                           4
Brunch restaurant                4
Coffee store                     3
Art cafe                         3
Night club                       3
Chicken restaurant               3
Fast food restaurant             3
Juice shop                       2
Donut shop                       2
Cake shop                        2
Coffee stand                     2
(kosong)                         2
Seblak restaurant                2
Soup kitchen                     2
Snack bar                        2
Children's cafe                  2
Noodle shop                      2
Steak house                      2
Food and beverage consultant     1
Coffee roastery                  1
Dessert shop                     1
Buffet restaurant      

## 5. Ekstrak `popular_times` menjadi `peak_day` dan `peak_hour`

`popular_times` berisi JSON `{hari: {jam: skor_keramaian_0_100}}`.
Hari dan jam dengan skor tertinggi dipilih sebagai puncak keramaian. Jika ada skor kembar, dipilih
hari paling awal (Senin dulu) lalu jam paling pagi. Baris dengan `popular_times` kosong (`{}`)
atau semua skor 0 diisi kosong (NaN).

In [7]:
def get_peak(value):
    """Kembalikan (peak_day, peak_hour) dari JSON popular_times."""
    if not isinstance(value, str) or value.strip() in ("", "{}"):
        return (np.nan, np.nan)
    try:
        data = json.loads(value)
    except (ValueError, TypeError):
        return (np.nan, np.nan)

    best_score, best_day, best_hour = 0, None, None
    for day in DAY_ORDER:
        hours = data.get(day) or {}
        for hour, score in sorted(hours.items(), key=lambda kv: int(kv[0])):
            if score is not None and score > best_score:   # '>' menjaga yang paling awal saat seri
                best_score, best_day, best_hour = score, day, int(hour)

    if best_day is None:
        return (np.nan, np.nan)
    return (best_day, best_hour)

peaks = [get_peak(v) for v in df["popular_times"]]
df["peak_day"] = [p[0] for p in peaks]
df["peak_hour"] = pd.array([p[1] for p in peaks], dtype="Int64")   # integer yang boleh kosong

print("Baris dengan peak terisi :", df["peak_day"].notna().sum())
print("Baris tanpa data keramaian:", df["peak_day"].isna().sum())
print()
print(df["peak_day"].value_counts().reindex(DAY_ORDER).to_string())
df[["title", "peak_day", "peak_hour"]].head()

Baris dengan peak terisi : 612
Baris tanpa data keramaian: 432

peak_day
Monday        45
Tuesday       60
Wednesday     59
Thursday      27
Friday        70
Saturday     294
Sunday        57


,title,peak_day,peak_hour
0,Kise Coffee,Thursday,14
1,Berdua Coffee,NaN,<NA>
2,Caffeinated,NaN,<NA>
3,DE VILLAGE CAFE,Saturday,20
4,All-Co Veteran,NaN,<NA>


## 6. Rapikan tipe data, validasi, dan ekspor

In [8]:
FINAL_COLUMNS = KEEP_COLUMNS + ["peak_day", "peak_hour"]

df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df["review_count"] = pd.to_numeric(df["review_count"], errors="coerce").fillna(0).astype(int)
df["review_rating"] = pd.to_numeric(df["review_rating"], errors="coerce")

# Rating 0 berarti belum ada ulasan -> jadikan kosong agar tidak menurunkan rata-rata
df.loc[df["review_count"] == 0, "review_rating"] = np.nan

df = df[FINAL_COLUMNS].reset_index(drop=True)

# ---- Validasi ----
assert df["place_id"].is_unique, "place_id masih ada yang duplikat"
assert df["address"].str.contains(KALSEL_PATTERN, case=False, regex=True).all(), "Ada alamat di luar Kalsel"
assert df["category"].str.strip().str.lower().isin(KEEP_CATEGORIES).all(), "Ada kategori di luar daftar"
assert df[["latitude", "longitude"]].notna().all().all(), "Ada koordinat kosong"
print("Semua validasi lolos.")

# ---- Ekspor ----
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
print(f"Tersimpan: {OUTPUT_FILE} ({len(df)} baris, {df.shape[1]} kolom)")
df.head()

Semua validasi lolos.
Tersimpan: clean_dataset_cafe_kalsel.csv (1044 baris, 12 kolom)


,place_id,title,category,address,latitude,longitude,open_hours,popular_times,review_count,review_rating,peak_day,peak_hour
0,ChIJHWziELEj5C0RjTyKdw1DrwA,Kise Coffee,Coffee shop,"Melayu (Di Arena Suzuki Bagian Kiri, Jl. A. Yani km 2 No.6, Melayu, Kec. Ban...",-3.324391,114.603467,"{""Friday"":[""9.00 am–12.00 am""],""Monday"":[""9.00 am–11.00 pm""],""Saturday"":[""9....","{""Friday"":{""10"":68,""11"":81,""12"":84,""13"":75,""14"":78,""15"":71,""16"":75,""17"":68,""...",153,4.8,Thursday,14
1,ChIJr_tWWgAj5C0RXmWeCegoXgY,Berdua Coffee,Coffee shop,"Jl. A. Yani No.147, Karang Mekar, Kec. Banjarmasin Tim., Kota Banjarmasin, K...",-3.327958,114.607845,"{""Friday"":[""7.30 pm–2.00 am""],""Monday"":[""7.30 pm–2.00 am""],""Saturday"":[""7.30...",{},1,5.0,NaN,<NA>
2,ChIJFRIbRAAj5C0RcjjW-dnqaic,Caffeinated,Coffee shop,"Jl. Manggis No.19a, RT.17/RW.02, Kuripan, Kec. Banjarmasin Tim., Kota Banjar...",-3.322415,114.610258,"{""Friday"":[""3.00 pm–1.00 am""],""Monday"":[""3.00 pm–12.00 am""],""Saturday"":[""3.0...",{},2,5.0,NaN,<NA>
3,ChIJ4yQ1orMj5C0R94uWD4vKKrc,DE VILLAGE CAFE,Cafe,"diatas Toko Tifa Home decor Bjm, Jl. Manggis No.121 Lantai 2 & 3, Kuripan, K...",-3.321195,114.610064,"{""Friday"":[""11.00 am–11.00 pm""],""Monday"":[""11.00 am–11.00 pm""],""Saturday"":[""...","{""Friday"":{""10"":0,""11"":33,""12"":37,""13"":40,""14"":42,""15"":48,""16"":57,""17"":72,""1...",137,4.3,Saturday,20
4,ChIJfRWTGw8j5C0RVbe95Oi1MAM,All-Co Veteran,Coffee shop,"Sebrang Pertigaan, Jl. Veteran Jl. Manggis, Sungai Bilu, Kec. Banjarmasin Ti...",-3.319982,114.609756,"{""Friday"":[""9.00 am–11.00 pm""],""Monday"":[""9.00 am–11.00 pm""],""Saturday"":[""9....",{},7,5.0,NaN,<NA>


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1044 entries, 0 to 1043
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   place_id       1044 non-null   str    
 1   title          1044 non-null   str    
 2   category       1044 non-null   str    
 3   address        1044 non-null   str    
 4   latitude       1044 non-null   float64
 5   longitude      1044 non-null   float64
 6   open_hours     1044 non-null   str    
 7   popular_times  1044 non-null   str    
 8   review_count   1044 non-null   int64  
 9   review_rating  972 non-null    float64
 10  peak_day       612 non-null    str    
 11  peak_hour      612 non-null    Int64  
dtypes: Int64(1), float64(3), int64(1), str(7)
memory usage: 99.0 KB
